# Random Forest

Il Random Forest è un algoritmo di machine learning supervisionato basato su una collezionei alberi decisionali indipendenti, combinati per migliorare la accuratezza e la robustezza rispetto a singoli alberi.

In [ ]:
import pandas as pd
import numpy as np
import math
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import f1_score, make_scorer
import tabulate
from pathlib import Path
import warnings
# Nascondo i warning
warnings.filterwarnings('ignore')

# Definisco il percorso dei file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Lista dei csv su cui fare training
datasets = {
    't2_medsam': FILE_PATH / 't2_medsam_masks.csv',
    't2_preprocessed': FILE_PATH / 't2_preprocessed_masks.csv',
    't2_original': FILE_PATH / 't2_original_masks.csv',
    'medsam_dynamic': FILE_PATH / 'medsam_dynamic.csv',
    'preprocessed_dynamic': FILE_PATH / 'preprocessed_dynamic.csv',
    'original_dynamic': FILE_PATH / 'original_dynamic.csv'
}

# Training

In [ ]:


def training(file_path, csv_name):
    # Legge il dataset
    df = pd.read_csv(file_path)

    # Definisco le colonne target
    original_target_list = ['PR [SII]', 'ER [SII]', 'KI67 [%]']

    # Vado a rimuovere le lesioni (righe) non valide
    df_validi = df.dropna(subset=original_target_list).copy()

    # Trasformo tutto in valori binari
    df_validi['PR_class'] = (df_validi['PR [SII]'] > 0.5).astype(int)
    df_validi['ER_class'] = (df_validi['ER [SII]'] > 0.5).astype(int)
    df_validi['KI67_class'] = (df_validi['KI67 [%]'] >= 20).astype(int)

    # Lista finale delle colonne target
    final_target_list = ['PR_class', 'ER_class', 'KI67_class']

    # Definisco le feature
    features_to_drop = ['Patient ID', 'lesion idx', 'tumor/benign', 'GRADE', 'isTN', 'Breast'] + original_target_list + final_target_list
    features = df_validi.drop(columns=features_to_drop, errors='ignore')

    # Target e Gruppi
    target = df_validi[final_target_list]
    groups = df_validi['Patient ID']

    # Riempio NaN
    features = features.fillna(features.mean())

    # Cross Validation
    cv = GroupKFold(n_splits=5)

    # Definisco il modello che uso
    base_model = RandomForestClassifier(random_state=42, n_jobs=1)
    multi_output_model = MultiOutputClassifier(base_model)

    iperparametri = {
      # Aumenta a 100-200. 
      # A differenza di XGBoost, il Random Forest NON va in overfitting se aumenti gli alberi.
      # Più alberi metti, più la predizione diventa stabile (media statistica migliore).
      'estimator__n_estimators': [100, 200],
      
      # CRITICO: Togli 'None'. 
      # 'None' lascia crescere l'albero all'infinito finché non isola ogni singolo paziente.
      # Con 82 pazienti, fermati a 3 o 5 per catturare solo le macro-regole.
      'estimator__max_depth': [3, 5],
      
      # Aumenta questo valore. 
      # Dice: "Non provare nemmeno a dividere un gruppo se sono meno di 10 persone".
      'estimator__min_samples_split': [5, 10],
      
      # CRITICO: Mai usare 1 con dataset piccoli.
      # Metti almeno 2 o 4. Significa che ogni foglia finale deve avere una "diagnosi"
      # basata su almeno 2-4 pazienti simili, non su un caso isolato.
      'estimator__min_samples_leaf': [2, 4],
      
      # 'sqrt' è lo standard (radice quadrata del num feature). 
      # 'log2' è ancora più restrittivo. Provali entrambi.
      'estimator__max_features': ['sqrt', 'log2'],
    }


    # Scorer Custom (zero_division=0 previene i NaN)
    def multi_f1_scorer(y_true, y_pred):
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        scores = []
        for i in range(y_true.shape[1]):
            scores.append(f1_score(y_true[:, i], y_pred[:, i], average='macro', zero_division=0))
        return np.mean(scores)

    scorer = make_scorer(multi_f1_scorer)

    # Calcolo combinazioni solo per info print
    total_combinations = math.prod(len(v) for v in iperparametri.values())
    print(f"\nInizio Grid Search ({total_combinations} combinazioni) per: {csv_name}")

    # Configurazione Grid Search
    grid_search = GridSearchCV(
        estimator=multi_output_model,
        param_grid=iperparametri,
        cv=cv,
        scoring=scorer,
        n_jobs=-1,     
        verbose=1,     
        refit=False,   
        error_score='raise'
    )

    # Esecuzione
    grid_search.fit(features, target, groups=groups)

  
    scores = []
    results = grid_search.cv_results_
    
    for i in range(len(results['params'])):
        params = {k.replace('estimator__', ''): v for k, v in results['params'][i].items()}
      
        current_fold_scores = [results[f'split{k}_test_score'][i] for k in range(5)]

        scores.append({
            **params,  # Spacchetta n_estimators, max_depth, ecc.
            'mean_score': results['mean_test_score'][i],
            'std_score': results['std_test_score'][i],
            'fold_scores': current_fold_scores
        })

    return scores


# Stampo i risultati in un formato piú leggibile

In [ ]:
def print_results(results_per_dataset):
    print("\n" + "=" * 80)
    print(" " * 25 + "RIEPILOGO DEI MIGLIORI RISULTATI")
    print("=" * 80)

    # Lista per il riepilogo finale comparativo
    summary_data = []

    for name, metrics_list in results_per_dataset.items():
        best_result = max(metrics_list, key=lambda x: x['mean_score'])

        print(f"\n{'─' * 80}")
        print(f" Dataset: {name}")
        print(f"{'─' * 80}")
        print(f"\n Performance: F1-score = {best_result['mean_score']:.3f} ± {best_result['std_score']:.3f}\n")

        # Tabella Iperparametri
        print("Iperparametri Ottimali:")
        params_table = [
            ['n_estimators', best_result['n_estimators']],
            ['max_depth', best_result['max_depth']],
            ['min_samples_split', best_result['min_samples_split']],
            ['min_samples_leaf', best_result['min_samples_leaf']],
            ['max_features', best_result['max_features']]
        ]
        print(tabulate.tabulate(params_table, headers=['Parametro', 'Valore'], tablefmt='simple'))
        print()

        # Aggiungi al riepilogo comparativo
        summary_data.append([
            name,
            f"{best_result['mean_score']:.3f}",
            f"{best_result['std_score']:.3f}",
            best_result['n_estimators'],
            best_result['max_depth'],
            best_result['max_features']
        ])

    # Riepilogo Comparativo Finale
    print("\n" + "=" * 80)
    print(" " * 25 + "CONFRONTO TRA TUTTI I DATASET")
    print("=" * 80 + "\n")

    # Ordina per F1-score decrescente
    summary_data.sort(key=lambda x: float(x[1]), reverse=True)

    print(tabulate.tabulate(summary_data,
                   headers=['Dataset', 'F1-score', 'Std Dev', 'N Est.', 'Max Depth', 'Max Features'],
                   tablefmt='grid',
                   floatfmt=('.3f', '.3f', '.3f', '.2f', 'g', '.1f', 'g')))

# Lettura dei file

In [ ]:
# Eseguo il training per tutti i dataset
results_per_dataset = {}

for name, file_path in datasets.items():
    results_per_dataset[name] = training(file_path, name)

# Stampo i risultati con tabulate
print_results(results_per_dataset)